*This is a Kaggle based notebook for testing Qwen3-VL*

### Dependencies And Installations

In [1]:
!pip install -q git+https://github.com/huggingface/transformers
!pip install -q qwen-vl-utils accelerate

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 618.0/618.0 kB 16.3 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 102.5 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.4/36.4 MB 54.8 MB/s eta 0:00:00:00:0100:01


In [ ]:
from huggingface_hub import login

# Paste your token inside the quotes
login(token="your_huggingface_token_here")

### Main Inference Code

In [3]:
import os
import torch
import re
import json
from PIL import Image
from transformers import Qwen3VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info

# ==========================================
# Global Variable for Downstream Cells
# ==========================================
# This will store the parsed JSON bounding boxes so you can access 
# it in the next Kaggle cells simply by calling `global_bbox_data`
global_bbox_data = None

# ==========================================
# 1. Initialize Model and Processor
# ==========================================
model_id = "Qwen/Qwen3-VL-8B-Instruct"

print(f"Loading {model_id} model... (this takes a few minutes)")
model = Qwen3VLForConditionalGeneration.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

processor = AutoProcessor.from_pretrained(model_id)
print("Model loaded successfully on Kaggle!")

# ==========================================
# 2. Core Inference Helper
# ==========================================
def run_vlm_inference(messages, max_new_tokens=512):
    """Processes messages, handles image scaling, and generates text."""
    text = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    image_inputs, video_inputs = process_vision_info(messages)
    
    inputs = processor(
        text=[text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt",
    ).to(model.device)

    with torch.no_grad():
        generated_ids = model.generate(
            **inputs, 
            max_new_tokens=max_new_tokens,
            temperature=0.1 # Low temp for factual extraction
        )
    
    generated_ids_trimmed = [
        out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
    ]
    output_text = processor.batch_decode(
        generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
    )
    return output_text[0]

# ==========================================
# 3. The Experiments 
# ==========================================
def run_experiment_1(full_frame_path):
    # Using string concatenation for the backticks to prevent markdown parser breakage
    prompt = (
        "Analyze this image and extract the following information. You MUST use this EXACT formatting:\n\n"
        '"Does this frame show at least one logo, brand name, or product?": [Yes/No]\n'
        '"Brand names and logos present": [Comma separated list or \'None\']\n'
        '"Bounding Boxes": \n'
        + "```" + "json\n" +
        "[\n"
        "  {\"brand\": \"Brand Name\", \"bbox_2d\": [ymin, xmin, ymax, xmax]}\n"
        "]\n"
        + "```\n" +
        '"Reasoning details": [Provide your reasoning for the detections]'
    )

    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": f"file://{full_frame_path}", "max_pixels": 1003520},
                {"type": "text", "text": prompt},
            ],
        }
    ]
    return run_vlm_inference(messages, max_new_tokens=512)

def run_experiment_2(full_frame_path, crop_path, brand_list=None):
    context_instruction = ""
    if brand_list:
        context_instruction = f"\nContext: The following brands were previously extracted from the full frame:\n{brand_list}\n"

    prompt = (
        "Image 1 is the full video frame.\n"
        f"Image 2 is a cropped region from that frame.{context_instruction}\n"
        "Task: Analyze the cropped region (Image 2) using the context of the full frame (Image 1). \n"
        "You MUST use this EXACT formatting:\n\n"
        '"Cropped region show:": [Full Logo / Partial Logo / Not a Logo]\n'
        '"Brand Name": [Name of the brand, or \'None\']\n'
        '"Reasoning details": [Provide a brief reasoning for your judgment]'
    )

    messages = [
        {
            "role": "user",
            "content": [
                {"type": "text", "text": "Image 1:"},
                {"type": "image", "image": f"file://{full_frame_path}", "max_pixels": 1003520},
                {"type": "text", "text": "Image 2:"},
                {"type": "image", "image": f"file://{crop_path}", "max_pixels": 250880}, 
                {"type": "text", "text": prompt},
            ],
        }
    ]
    return run_vlm_inference(messages, max_new_tokens=512)

# ==========================================
# 4. Execution / Testing Block
# ==========================================
if __name__ == "__main__":
    # --- Take User Input Immediately ---
    user_choice = input("Enter '1' to provide the Experiment 1 list to Experiment 2, or '0' to not provide it: ").strip()
    use_list = True if user_choice == '1' else False
    
    # --- Kaggle Working Directory Paths ---
    full_frame = "/kaggle/input/datasets/mhd01ali/logo-crops-subset/Logo-Subset/Frame-4/Screenshot 2026-03-23 at 3.42.15PM.png" 
    cropped_region = "/kaggle/input/datasets/mhd01ali/logo-crops-subset/Logo-Subset/Crops-4/336_10_video_3 (online-video-cutter.com).png"
    if not os.path.exists(full_frame):
        # Create dummy images to test pipeline if paths don't exist
        Image.new('RGB', (1920, 1080), color='white').save(full_frame)
        Image.new('RGB', (200, 200), color='blue').save(cropped_region)

    print("\n" + "="*40)
    print("EXPERIMENT 1:")
    print("="*40)
    exp1_output = run_experiment_1(full_frame)
    print(exp1_output)
    
    # --- Extract and Save JSON to Global Variable ---
    # Look for the markdown code block containing the JSON
    json_match = re.search(r'```json\s*(.*?)\s*```', exp1_output, re.DOTALL)
    if json_match:
        try:
            global_bbox_data = json.loads(json_match.group(1))
            print("\n[System]: Bounding boxes successfully parsed and saved to 'global_bbox_data'.")
        except json.JSONDecodeError:
            print("\n[System]: Failed to parse JSON from Experiment 1 output. Verify model output format.")
    else:
        print("\n[System]: No JSON block found in Experiment 1 output.")
    
    print("\n" + "="*40)
    print("EXPERIMENT 2:")
    print("="*40)
    
    # Extract just the brand list string to pass to Exp 2 if requested
    passed_list = None
    if use_list:
        # Regex to capture everything after "Brand names and logos present": until the next line break
        list_match = re.search(r'"Brand names and logos present":\s*(.*)', exp1_output)
        if list_match:
            passed_list = list_match.group(1).strip()
    
    exp2_output = run_experiment_2(full_frame, cropped_region, brand_list=passed_list)
    print(exp2_output)

Loading Qwen/Qwen3-VL-8B-Instruct model... (this takes a few minutes)


config.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/750 [00:00<?, ?it/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

video_preprocessor_config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

Model loaded successfully on Kaggle!


Enter '1' to provide the Experiment 1 list to Experiment 2, or '0' to not provide it:  1



EXPERIMENT 1:
"Does this frame show at least one logo, brand name, or product?": Yes
"Brand names and logos present": Rai Sport HD, neo Borocillina
"Bounding Boxes": 
```json
[
  {"brand": "Rai Sport HD", "bbox_2d": [47, 92, 166, 148]},
  {"brand": "neo Borocillina", "bbox_2d": [328, 170, 652, 285]}
]
```
"Reasoning details": The image is an advertisement for "neo Borocillina" products, which is clearly visible as the main brand name in large text. The "Rai Sport HD" logo is present in the top-left corner, indicating the broadcasting channel. Both are identifiable brand names/logos. The bounding boxes are estimated based on their visible positions in the frame.

[System]: Bounding boxes successfully parsed and saved to 'global_bbox_data'.

EXPERIMENT 2:
"Cropped region show:": Partial Logo
"Brand Name": neo
"Reasoning details": The cropped region shows only the word "neo", which is part of the larger brand name "neo Borocillina" visible in the full frame. It is not a complete logo on 

In [ ]:
print(global_bbox_data)

### Qwen Code Interim (Experiment 1/2)

In [ ]:
import os
import torch
import re
from PIL import Image
import json
from transformers import Qwen3VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info

# ==========================================
# 1. Initialize Model and Processor
# ==========================================
model_id = "Qwen/Qwen3-VL-8B-Instruct"

print(f"Loading {model_id} model... (this takes a few minutes)")
model = Qwen3VLForConditionalGeneration.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

processor = AutoProcessor.from_pretrained(model_id)
print("Model loaded successfully on Kaggle!")

# ==========================================
# 2. Core Inference Helper
# ==========================================
def run_vlm_inference(messages, max_new_tokens=512):
    """Processes messages, handles image scaling, and generates text."""
    text = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    image_inputs, video_inputs = process_vision_info(messages)
    
    inputs = processor(
        text=[text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt",
    ).to(model.device)

    with torch.no_grad():
        generated_ids = model.generate(
            **inputs, 
            max_new_tokens=max_new_tokens,
            temperature=0.1 # Low temp for factual extraction
        )
    
    generated_ids_trimmed = [
        out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
    ]
    output_text = processor.batch_decode(
        generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
    )
    return output_text[0]

# ==========================================
# 3. The Experiments 
# ==========================================
def run_experiment_1(full_frame_path):
    prompt = """Analyze this image and provide the following clearly:
1. Does this frame show at least one logo, brand name, or product? (Answer Yes or No)
2. Extract a list of all brand names and logos present.
3. Provide the bounding box coordinates for each detected brand/logo.
4. Provide reasoning details for your detections."""

    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": f"file://{full_frame_path}", "max_pixels": 1003520},
                {"type": "text", "text": prompt},
            ],
        }
    ]
    return run_vlm_inference(messages, max_new_tokens=512)

def run_experiment_2(full_frame_path, crop_path, brand_list=None):
    context_instruction = ""
    if brand_list:
        context_instruction = f"\nContext: The following brands were previously extracted from the full frame:\n{brand_list}\n"

    prompt = f"""Image 1 is the full video frame.
Image 2 is a cropped region from that frame.{context_instruction}
Task: Analyze the cropped region (Image 2) using the context of the full frame (Image 1) and answer the following:
1. Does the cropped region show a 'Full Logo', a 'Partial Logo', or 'Not a Logo'?
2. What is the brand name? (If none, say 'None')
3. Provide a brief reasoning for your judgment."""

    messages = [
        {
            "role": "user",
            "content": [
                {"type": "text", "text": "Image 1:"},
                {"type": "image", "image": f"file://{full_frame_path}", "max_pixels": 1003520},
                {"type": "text", "text": "Image 2:"},
                {"type": "image", "image": f"file://{crop_path}", "max_pixels": 250880}, 
                {"type": "text", "text": prompt},
            ],
        }
    ]
    return run_vlm_inference(messages, max_new_tokens=512)

# ==========================================
# 4. Helper: Parse Bounding Boxes to Pixels (Optional Use)
# ==========================================
def parse_and_scale_bboxes(model_output, image_path):
    """Handles both Qwen's native tag format AND generated JSON format."""
    try:
        with Image.open(image_path) as img:
            width, height = img.size
    except Exception as e:
        return f"Error opening image: {e}"

    parsed_results = []
    
    if "```json" in model_output or model_output.strip().startswith('['):
        try:
            clean_json = model_output.replace('```json', '').replace('```', '').strip()
            boxes = json.loads(clean_json)
            for box in boxes:
                if 'bbox_2d' in box:
                    xmin_norm, ymin_norm, xmax_norm, ymax_norm = box['bbox_2d']
                    x_min = int((xmin_norm / 1000) * width)
                    y_min = int((ymin_norm / 1000) * height)
                    x_max = int((xmax_norm / 1000) * width)
                    y_max = int((ymax_norm / 1000) * height)
                    parsed_results.append({
                        "label": box.get('label', 'Unknown Brand'),
                        "bbox_pixels": [x_min, y_min, x_max, y_max]
                    })
            return parsed_results
        except Exception:
            pass # Fall back to regex

    box_pattern = re.compile(r'\[(\d+),\s*(\d+),\s*(\d+),\s*(\d+)\]')
    lines = model_output.split('\n')
    for line in lines:
        match = box_pattern.search(line)
        if match:
            ymin_norm, xmin_norm, ymax_norm, xmax_norm = map(int, match.groups())
            x_min = int((xmin_norm / 1000) * width)
            y_min = int((ymin_norm / 1000) * height)
            x_max = int((xmax_norm / 1000) * width)
            y_max = int((ymax_norm / 1000) * height)
            
            label = line[:match.start()].replace('<|object_ref_start|>', '').replace('<|object_ref_end|>', '').replace('<|box_start|>', '').strip(' :*')
            if not label: label = "Unknown Brand"
            parsed_results.append({
                "label": label,
                "bbox_pixels": [x_min, y_min, x_max, y_max]
            })
            
    return parsed_results

# ==========================================
# 5. Execution / Testing Block
# ==========================================
if __name__ == "__main__":
    # --- Take User Input Immediately ---
    user_choice = input("Enter '1' to provide the Experiment 1 list to Experiment 2, or '0' to not provide it: ").strip()
    use_list = True if user_choice == '1' else False
    
    # --- Kaggle Working Directory Paths ---
    full_frame = "/kaggle/input/datasets/mhd01ali/logo-crops-subset/Logo-Subset/Frame-1/Screenshot 2026-03-23 at 3.30.09PM.png" 
    cropped_region = "/kaggle/input/datasets/mhd01ali/logo-crops-subset/Logo-Subset/Crops-1/34_10_video_3 (online-video-cutter.com).png"
    
    if not os.path.exists(full_frame):
        # Create dummy images to test pipeline if paths don't exist
        Image.new('RGB', (1920, 1080), color='white').save(full_frame)
        Image.new('RGB', (200, 200), color='blue').save(cropped_region)

    print("\n" + "="*40)
    print("EXPERIMENT 1:")
    print("="*40)
    exp1_output = run_experiment_1(full_frame)
    print(exp1_output)
    
    print("\n" + "="*40)
    print("EXPERIMENT 2:")
    print("="*40)
    
    # Only pass the output of exp1 as the list if the user selected '1'
    passed_list = exp1_output if use_list else None
    
    exp2_output = run_experiment_2(full_frame, cropped_region, brand_list=passed_list)
    print(exp2_output)
    
    # Optional: If you want to parse out just the pixels from Exp 1 later, you can still use the helper:
    # parsed_boxes = parse_and_scale_bboxes(exp1_output, full_frame)

### Old Qwen Inference

In [ ]:
import os
import torch
import re
from PIL import Image
import json
# Using AutoModelForCausalLM to seamlessly map to the new Qwen3VL architecture
from transformers import Qwen3VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info

# ==========================================
# 1. Initialize Model and Processor
# ==========================================
# Strictly Qwen 3 VL 8B. Unquantized bfloat16 utilizes ~16GB of your 32GB VRAM.
model_id = "Qwen/Qwen3-VL-8B-Instruct"

print(f"Loading {model_id} model... (this takes a few minutes)")
model = Qwen3VLForConditionalGeneration.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

processor = AutoProcessor.from_pretrained(model_id)
print("Model loaded successfully on Kaggle!")

# ==========================================
# 2. Core Inference Helper
# ==========================================
def run_vlm_inference(messages, max_new_tokens=256):
    """Processes messages, handles image scaling, and generates text."""
    text = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    image_inputs, video_inputs = process_vision_info(messages)
    
    inputs = processor(
        text=[text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt",
    ).to(model.device)

    with torch.no_grad():
        generated_ids = model.generate(
            **inputs, 
            max_new_tokens=max_new_tokens,
            temperature=0.1 # Low temp for factual extraction
        )
    
    generated_ids_trimmed = [
        out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
    ]
    output_text = processor.batch_decode(
        generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
    )
    return output_text[0]

# ==========================================
# 3. The Four Experiments (VRAM-Optimized)
# ==========================================

def detect_any_logo(full_frame_path):
    messages = [
        {
            "role": "user",
            "content": [
                # Scaled back to safely fit in the T4's attention memory buffer
                {"type": "image", "image": f"file://{full_frame_path}", "max_pixels": 1003520},
                {"type": "text", "text": "Does this image show at least one logo, brand name, or product? Answer strictly with 'Yes' or 'No'."},
            ],
        }
    ]
    return run_vlm_inference(messages, max_new_tokens=10)

def extract_brand_list(full_frame_path):
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": f"file://{full_frame_path}", "max_pixels": 1003520},
                {"type": "text", "text": "Extract a list of all brand names, companies, products, and logos shown in this image. Format the output as a comma-separated list. If none are found, say 'None'."},
            ],
        }
    ]
    return run_vlm_inference(messages)

def identify_cropped_region(full_frame_path, crop_path, extracted_brands, audio_brands):
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "text", "text": "Image 1 is the full video frame:"},
                {"type": "image", "image": f"file://{full_frame_path}", "max_pixels": 1003520},
                {"type": "text", "text": "Image 2 is a cropped region from that frame:"},
                {"type": "image", "image": f"file://{crop_path}", "max_pixels": 250880}, 
                {"type": "text", "text": f"""
Context: 
- Brands previously extracted from the full frame: [{extracted_brands}]
- Brands mentioned in the audio: [{audio_brands}]

Task: Analyze the cropped region (Image 2) using the context of the full frame (Image 1). 
Follow this strict logic to determine your output:
1. If the cropped region clearly shows a logo or brand that matches one of the items in the provided Context lists, output ONLY that brand name.
2. If the cropped region clearly shows a logo or brand that is NOT in the Context lists, output ONLY that new brand name.
3. If the cropped region does NOT show any recognizable logo, brand, company, or product (e.g., it is just a generic object, a person, text that isn't a brand, or background), output strictly 'None'.
"""},
            ],
        }
    ]
    return run_vlm_inference(messages)

def extract_brand_bboxes(full_frame_path):
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": f"file://{full_frame_path}", "max_pixels": 1003520},
                {"type": "text", "text": "Detect all brands, companies, products, and logos in this image. Output their names and bounding boxes."},
            ],
        }
    ]
    return run_vlm_inference(messages, max_new_tokens=512)

# ==========================================
# 4. Helper: Parse Bounding Boxes to Pixels
# ==========================================
def parse_and_scale_bboxes(model_output, image_path):
    """
    Handles both Qwen's native tag format AND generated JSON format.
    Converts 1000-normalized coordinates to actual image pixels.
    """
    try:
        with Image.open(image_path) as img:
            width, height = img.size
    except Exception as e:
        return f"Error opening image: {e}"

    parsed_results = []

    # ---------------------------------------------------------
    # SCENARIO A: The model output a JSON array (like in your test)
    # ---------------------------------------------------------
    if "```json" in model_output or model_output.strip().startswith('['):
        try:
            # Clean markdown formatting if present
            clean_json = model_output.replace('```json', '').replace('```', '').strip()
            boxes = json.loads(clean_json)
            
            for box in boxes:
                if 'bbox_2d' in box:
                    # JSON bboxes are usually [xmin, ymin, xmax, ymax]
                    xmin_norm, ymin_norm, xmax_norm, ymax_norm = box['bbox_2d']
                    
                    x_min = int((xmin_norm / 1000) * width)
                    y_min = int((ymin_norm / 1000) * height)
                    x_max = int((xmax_norm / 1000) * width)
                    y_max = int((ymax_norm / 1000) * height)
                    
                    parsed_results.append({
                        "label": box.get('label', 'Unknown Brand'),
                        "bbox_pixels": [x_min, y_min, x_max, y_max]
                    })
            return parsed_results
        except Exception as e:
            print(f"JSON Parsing failed, falling back to Regex. Error: {e}")

    # ---------------------------------------------------------
    # SCENARIO B: The model output native Qwen tags
    # ---------------------------------------------------------
    box_pattern = re.compile(r'\[(\d+),\s*(\d+),\s*(\d+),\s*(\d+)\]')
    lines = model_output.split('\n')
    
    for line in lines:
        match = box_pattern.search(line)
        if match:
            # Native Qwen tags are [ymin, xmin, ymax, xmax]
            ymin_norm, xmin_norm, ymax_norm, xmax_norm = map(int, match.groups())
            
            x_min = int((xmin_norm / 1000) * width)
            y_min = int((ymin_norm / 1000) * height)
            x_max = int((xmax_norm / 1000) * width)
            y_max = int((ymax_norm / 1000) * height)
            
            label = line[:match.start()].replace('<|object_ref_start|>', '').replace('<|object_ref_end|>', '').replace('<|box_start|>', '').strip(' :*')
            if not label: label = "Unknown Brand"
            
            parsed_results.append({
                "label": label,
                "bbox_pixels": [x_min, y_min, x_max, y_max]
            })
            
    return parsed_results

# ==========================================
# 5. Execution / Testing Block
# ==========================================
if __name__ == "__main__":
    # --- Kaggle Working Directory Paths ---
    full_frame = "/kaggle/input/datasets/mhd01ali/logo-crops-subset/Logo-Subset/Frame-1/Screenshot 2026-03-23 at 3.30.09PM.png" 
    cropped_region = "/kaggle/input/datasets/mhd01ali/logo-crops-subset/Logo-Subset/Crops-1/34_10_video_3 (online-video-cutter.com).png"
    
    if not os.path.exists(full_frame):
        # Create a dummy 1920x1080 image to test processing
        Image.new('RGB', (1920, 1080), color='white').save(full_frame)
        Image.new('RGB', (200, 200), color='blue').save(cropped_region)

    print("\nStarting Pipeline Tests...\n" + "-"*30)

    # 1. Binary
    print("\n[Exp 1] Binary Logo Detection:")
    print("Result:", detect_any_logo(full_frame))
    
    # 2. List
    print("\n[Exp 2] Extracted Brands List:")
    brands_list = extract_brand_list(full_frame)
    print("Result:", brands_list)
    
    # 3. Contextual Crop
    print("\n[Exp 3] Region Identification (Crop):")
    audio_context = "Nike, Apple" 
    crop_result = identify_cropped_region(full_frame, cropped_region, brands_list, audio_context)
    print("Result:", crop_result)
    
    # 4. Bounding Boxes
    print("\n[Exp 4] Bounding Boxes Generation:")
    raw_bbox_output = extract_brand_bboxes(full_frame)
    print("Raw Model Output:\n", raw_bbox_output)
    
    print("\n[Helper] Parsed Pixel Coordinates:")
    parsed_boxes = parse_and_scale_bboxes(raw_bbox_output, full_frame)
    for item in parsed_boxes:
        print(f"- {item['label']}: {item['bbox_pixels']}")